In [2]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher

# Load the completed Excel file
file_path = "/content/cv_llm4_ground_truth_10_each.xlsx"

normal_df = pd.read_excel(file_path, sheet_name="Normal")
damaged_df = pd.read_excel(file_path, sheet_name="Simulate or use distortions")

print("Normal shape:", normal_df.shape)
print("Damaged shape:", damaged_df.shape)

print("\nNormal columns:")
print(normal_df.columns.tolist())

print("\nDamaged columns:")
print(damaged_df.columns.tolist())

Normal shape: (61, 8)
Damaged shape: (90, 9)

Normal columns:
['Case_ID', 'Plate_Prediction', 'Readability', 'Visual_Issue', 'Explanation', 'Confidence_0_to_100', 'Original_Image', 'Ground_Truth']

Damaged columns:
['Case_ID', 'Plate_Prediction', 'Readability', 'Visual_Issue', 'Explanation', 'Confidence_0_to_100', 'Distortion_Type', 'Original_Image', 'Ground_Truth']


In [3]:
damaged_counts = damaged_df["Distortion_Type"].value_counts().sort_index()
print(damaged_counts)

# Check if every distortion type has exactly 10 rows
all_10 = (damaged_counts == 10).all()
print("\nEvery distortion type has 10 rows:", all_10)

Distortion_Type
Dirty            10
Gaussian_Blur    10
Low_Light        10
Low_Res          10
Motion_Blur      10
Occlusion        10
Over_Exposed     10
Rotation         10
Tilted           10
Name: count, dtype: int64

Every distortion type has 10 rows: True


In [4]:
def normalize_plate(value):
    if pd.isna(value):
        return ""

    value = str(value).upper()

    # Treat unreadable / unclear predictions as empty
    unreadable_words = ["UNREADABLE", "NOT READABLE", "UNCLEAR", "N/A", "NA", "NONE"]
    if value.strip() in unreadable_words:
        return ""

    # Keep only A-Z and 0-9
    value = re.sub(r"[^A-Z0-9]", "", value)

    return value

In [5]:
def char_similarity(pred, gt):
    pred = normalize_plate(pred)
    gt = normalize_plate(gt)

    if gt == "":
        return np.nan

    return SequenceMatcher(None, pred, gt).ratio()

In [6]:
def add_accuracy_columns(df):
    df = df.copy()

    df["Prediction_Clean"] = df["Plate_Prediction"].apply(normalize_plate)
    df["Ground_Truth_Clean"] = df["Ground_Truth"].apply(normalize_plate)

    # Valid rows are rows where ground truth exists
    df["Has_Ground_Truth"] = df["Ground_Truth_Clean"] != ""

    # Exact match: prediction must fully equal ground truth
    df["Exact_Match"] = df["Prediction_Clean"] == df["Ground_Truth_Clean"]

    # If ground truth is missing, do not count it as correct or wrong
    df.loc[df["Has_Ground_Truth"] == False, "Exact_Match"] = np.nan

    # Character-level similarity
    df["Char_Similarity"] = df.apply(
        lambda row: char_similarity(row["Plate_Prediction"], row["Ground_Truth"]),
        axis=1
    )

    return df

normal_eval = add_accuracy_columns(normal_df)
damaged_eval = add_accuracy_columns(damaged_df)

/tmp/ipykernel_6107/3634477024.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df["Has_Ground_Truth"] == False, "Exact_Match"] = np.nan
/tmp/ipykernel_6107/3634477024.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df["Has_Ground_Truth"] == False, "Exact_Match"] = np.nan


In [7]:
normal_valid = normal_eval[normal_eval["Has_Ground_Truth"] == True]

normal_accuracy = normal_valid["Exact_Match"].mean()
normal_char_similarity = normal_valid["Char_Similarity"].mean()

print("Normal valid rows:", len(normal_valid))
print("Normal exact-match accuracy:", round(normal_accuracy * 100, 2), "%")
print("Normal average character similarity:", round(normal_char_similarity * 100, 2), "%")

Normal valid rows: 46
Normal exact-match accuracy: 91.3 %
Normal average character similarity: 98.06 %


In [8]:
damaged_summary = damaged_eval.groupby("Distortion_Type").agg(
    Total_Images=("Case_ID", "count"),
    Valid_Ground_Truth=("Has_Ground_Truth", "sum"),
    Exact_Correct=("Exact_Match", "sum"),
    Exact_Accuracy=("Exact_Match", "mean"),
    Avg_Char_Similarity=("Char_Similarity", "mean"),
    Avg_Confidence=("Confidence_0_to_100", "mean")
).reset_index()

damaged_summary["Exact_Accuracy_%"] = damaged_summary["Exact_Accuracy"] * 100
damaged_summary["Avg_Char_Similarity_%"] = damaged_summary["Avg_Char_Similarity"] * 100

damaged_summary = damaged_summary[
    [
        "Distortion_Type",
        "Total_Images",
        "Valid_Ground_Truth",
        "Exact_Correct",
        "Exact_Accuracy_%",
        "Avg_Char_Similarity_%",
        "Avg_Confidence"
    ]
]

damaged_summary

,Distortion_Type,Total_Images,Valid_Ground_Truth,Exact_Correct,Exact_Accuracy_%,Avg_Char_Similarity_%,Avg_Confidence
0,Dirty,10,10,10,100.0,100.000000,90.3
1,Gaussian_Blur,10,10,1,10.0,25.238095,20.1
2,Low_Light,10,10,10,100.0,100.000000,85.6
3,Low_Res,10,10,3,30.0,36.666667,31.0
4,Motion_Blur,10,10,1,10.0,25.844156,19.7
5,Occlusion,10,10,9,90.0,99.090909,87.5
6,Over_Exposed,10,10,9,90.0,97.272727,89.1
7,Rotation,10,10,9,90.0,98.571429,91.1
8,Tilted,10,10,8,80.0,97.142857,84.2


In [9]:
damaged_valid = damaged_eval[damaged_eval["Has_Ground_Truth"] == True]

damaged_accuracy = damaged_valid["Exact_Match"].mean()
damaged_char_similarity = damaged_valid["Char_Similarity"].mean()

print("Damaged valid rows:", len(damaged_valid))
print("Damaged exact-match accuracy:", round(damaged_accuracy * 100, 2), "%")
print("Damaged average character similarity:", round(damaged_char_similarity * 100, 2), "%")

Damaged valid rows: 90
Damaged exact-match accuracy: 66.67 %
Damaged average character similarity: 75.54 %


In [10]:
normal_summary = pd.DataFrame({
    "Distortion_Type": ["Normal"],
    "Total_Images": [len(normal_eval)],
    "Valid_Ground_Truth": [normal_eval["Has_Ground_Truth"].sum()],
    "Exact_Correct": [normal_eval["Exact_Match"].sum()],
    "Exact_Accuracy_%": [normal_accuracy * 100],
    "Avg_Char_Similarity_%": [normal_char_similarity * 100],
    "Avg_Confidence": [normal_eval["Confidence_0_to_100"].mean()]
})

final_summary = pd.concat([normal_summary, damaged_summary], ignore_index=True)

final_summary

,Distortion_Type,Total_Images,Valid_Ground_Truth,Exact_Correct,Exact_Accuracy_%,Avg_Char_Similarity_%,Avg_Confidence
0,Normal,61,46,42,91.304348,98.061359,84.934426
1,Dirty,10,10,10,100.0,100.000000,90.300000
2,Gaussian_Blur,10,10,1,10.0,25.238095,20.100000
3,Low_Light,10,10,10,100.0,100.000000,85.600000
4,Low_Res,10,10,3,30.0,36.666667,31.000000
5,Motion_Blur,10,10,1,10.0,25.844156,19.700000
6,Occlusion,10,10,9,90.0,99.090909,87.500000
7,Over_Exposed,10,10,9,90.0,97.272727,89.100000
8,Rotation,10,10,9,90.0,98.571429,91.100000
9,Tilted,10,10,8,80.0,97.142857,84.200000
